In [ ]:
import pandas as pd
import joblib
import os
import re # For basic text cleaning
import logging
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Configuration ---
DATA_DIR = "./data"
READMISSION_DATA_FILE = os.path.join(DATA_DIR, "readmission_data.csv")
CLINICAL_NOTES_FILE = os.path.join(DATA_DIR, "clinical_notes_data.csv")
MEDICAL_IMAGING_FILE = os.path.join(DATA_DIR, "medical_imaging_data.csv")

#Associated Columns 
TARGET_COLUMN = "readmission"
JOIN_KEY = "patient_id" 
NOTES_TEXT_COLUMN = "note_text" 
IMAGING_SCAN_TYPE_COLUMN = "modality" 

# Features from readmission_data.csv
BASE_NUMERICAL_FEATURES = [
    "age", "systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
    "temperature", "oxygen_saturation", "glucose", "hemoglobin",
    "white_blood_cells", "platelet_count", "sodium", "potassium",
    "creatinine", "length_of_stay"
]
BASE_CATEGORICAL_FEATURES = [
    "hypertension", "diabetes", "coronary_artery_disease", "heart_failure",
    "stroke_history", "copd"
]

# New features to be created
NOTES_FEATURE_NAME = 'aggregated_notes' 
IMAGING_COUNT_FEATURE = 'imaging_count' 
IMAGING_FLAG_FEATURES = ['has_ct', 'has_mri', 'has_xray'] 

MODEL_OUTPUT_FILENAME = "model.pkl" 

# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Helper Functions ---
def clean_text(text):
    """Basic text cleaning."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text

# --- Main Training Logic ---
if __name__ == "__main__":
    logging.info("Starting extended training script...")

    #Load Data
    logging.info("Loading data...")
    try:
        df_readmission = pd.read_csv(READMISSION_DATA_FILE)
        df_notes = pd.read_csv(CLINICAL_NOTES_FILE)
        df_imaging = pd.read_csv(MEDICAL_IMAGING_FILE)
        logging.info("Data loaded successfully.")
    except FileNotFoundError as e:
        logging.error(f"Error loading data: {e}. Please check file paths.")
        exit()
    except Exception as e:
        logging.error(f"Error loading data: {e}")
        exit()

    #Preprocess & Aggregate Notes ---
    logging.info("Preprocessing clinical notes...")
    if JOIN_KEY not in df_notes.columns or NOTES_TEXT_COLUMN not in df_notes.columns:
        logging.error(f"Error: Notes CSV must contain '{JOIN_KEY}' and '{NOTES_TEXT_COLUMN}'.")
        exit()
    # Clean and aggregate notes per encounter
    df_notes[NOTES_TEXT_COLUMN] = df_notes[NOTES_TEXT_COLUMN].apply(clean_text)
    notes_agg = df_notes.groupby(JOIN_KEY)[NOTES_TEXT_COLUMN].apply(lambda x: ' '.join(x)).reset_index()
    notes_agg.rename(columns={NOTES_TEXT_COLUMN: NOTES_FEATURE_NAME}, inplace=True)
    logging.info(f"Aggregated notes. Shape: {notes_agg.shape}")

    #Preprocess & Aggregate Imaging Data (Metadata Example) ---
    logging.info("Preprocessing imaging data (metadata)...")
    if JOIN_KEY not in df_imaging.columns or IMAGING_SCAN_TYPE_COLUMN not in df_imaging.columns:
        logging.error(f"Error: Imaging CSV must contain '{JOIN_KEY}' and '{IMAGING_SCAN_TYPE_COLUMN}'.")
        exit()

    # Calculate image count per encounter
    imaging_counts = df_imaging.groupby(JOIN_KEY).size().reset_index(name=IMAGING_COUNT_FEATURE)

    # Create flags for specific scan types (example)
    df_imaging[IMAGING_SCAN_TYPE_COLUMN] = df_imaging[IMAGING_SCAN_TYPE_COLUMN].str.upper() # Normalize
    imaging_flags = pd.get_dummies(df_imaging[[JOIN_KEY, IMAGING_SCAN_TYPE_COLUMN]], columns=[IMAGING_SCAN_TYPE_COLUMN], prefix='scan')
    # Keep only flags we defined and aggregate by encounter_id (take max, assumes 0/1)
    desired_flag_cols = ['scan_CT', 'scan_MRI', 'scan_XRAY'] # Corresponds to IMAGING_FLAG_FEATURES
    for col in desired_flag_cols:
        if col not in imaging_flags.columns:
            imaging_flags[col] = 0 # Add column if a scan type doesn't exist in data
    imaging_flags = imaging_flags.groupby(JOIN_KEY)[desired_flag_cols].max().reset_index()
    imaging_flags.rename(columns={'scan_CT': 'has_ct', 'scan_MRI': 'has_mri', 'scan_XRAY': 'has_xray'}, inplace=True)

    # Combine imaging features
    imaging_agg = pd.merge(imaging_counts, imaging_flags, on=JOIN_KEY, how='outer')
    logging.info(f"Aggregated imaging features. Shape: {imaging_agg.shape}")


    #Join DataFrames ---
    logging.info("Joining datasets...")
    # Start with readmission data (should have the target)
    df_merged = pd.merge(df_readmission, notes_agg, on=JOIN_KEY, how='left')
    df_merged = pd.merge(df_merged, imaging_agg, on=JOIN_KEY, how='left')

    # Fill NaNs created by left joins
    df_merged[NOTES_FEATURE_NAME].fillna("", inplace=True) # Fill missing notes with empty string
    df_merged[IMAGING_COUNT_FEATURE].fillna(0, inplace=True) # Fill missing image count with 0
    for flag_col in IMAGING_FLAG_FEATURES:
         df_merged[flag_col].fillna(0, inplace=True) # Fill missing flags with 0 (False)
    logging.info(f"Merged data shape: {df_merged.shape}")


    #Data Preparation  ---
    if TARGET_COLUMN not in df_merged.columns:
        logging.error(f"Error: Target column '{TARGET_COLUMN}' not found after merge.")
        exit()
    # Check for base features
    missing_base_features = [f for f in BASE_NUMERICAL_FEATURES + BASE_CATEGORICAL_FEATURES if f not in df_merged.columns]
    if missing_base_features:
         logging.error(f"Error: Base features missing after merge: {missing_base_features}")
         exit()

    initial_rows = len(df_merged)
    df_merged.dropna(subset=[TARGET_COLUMN], inplace=True)
    if len(df_merged) < initial_rows:
        logging.warning(f"Dropped {initial_rows - len(df_merged)} rows due to missing target.")
    if len(df_merged) == 0:
        logging.error("No data remaining.")
        exit()

    # Define final feature lists including new ones
    FINAL_NUMERICAL_FEATURES = BASE_NUMERICAL_FEATURES + [IMAGING_COUNT_FEATURE]
    FINAL_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES + IMAGING_FLAG_FEATURES
    ALL_FEATURES_FOR_MODEL = FINAL_NUMERICAL_FEATURES + FINAL_CATEGORICAL_FEATURES + [NOTES_FEATURE_NAME]

    X = df_merged[ALL_FEATURES_FOR_MODEL]
    y = df_merged[TARGET_COLUMN]
    logging.info(f"Final Features shape: {X.shape}, Target shape: {y.shape}")

    # Split Data
    logging.info("Splitting data...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    #Update Preprocessing Pipeline ---
    logging.info("Defining updated preprocessing steps...")
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    # TF-IDF for the aggregated notes text
    # Hyperparameters (max_features, ngram_range, stopwords)
    text_transformer = TfidfVectorizer(max_features=5000, 
                                       ngram_range=(1, 2),
                                       stop_words='english')

    # Update ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, FINAL_NUMERICAL_FEATURES),
            ('cat', categorical_transformer, FINAL_CATEGORICAL_FEATURES),
            ('text', text_transformer, NOTES_FEATURE_NAME) # Apply TF-IDF to the notes column
        ],
        remainder='drop', 
        n_jobs=-1 
        )

    #Define Model ---
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
    logging.info(f"Using model: {type(model).__name__}")

    #Create Full Training Pipeline ---
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    logging.info("Complete extended pipeline created.")

    #Train the Model ---
    logging.info("Training the extended pipeline...")
    try:
        pipeline.fit(X_train, y_train)
        logging.info("Pipeline training completed.")
    except Exception as e:
        logging.error(f"Error during pipeline training: {e}", exc_info=True)
        exit()

    #Evaluate the Model ---
    logging.info("Evaluating model performance on the test set...")
    y_pred = pipeline.predict(X_test)
    # Handle cases where predict_proba might not be available or desired
    try:
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1] # Probability of positive class
        auc = roc_auc_score(y_test, y_pred_proba)
        logging.info(f"Test Set AUC: {auc:.4f}")
    except AttributeError:
        logging.warning("Model does not support predict_proba. AUC not calculated.")
    except Exception as e:
        logging.warning(f"Could not calculate AUC: {e}")


    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    logging.info(f"Test Set Accuracy: {accuracy:.4f}")
    logging.info("Test Set Classification Report:\n" + report)

    # --- 11. Save the Pipeline ---
    logging.info(f"Saving the trained pipeline to {MODEL_OUTPUT_FILENAME}...")
    try:
        joblib.dump(pipeline, MODEL_OUTPUT_FILENAME)
        logging.info("Pipeline saved successfully.")
    except Exception as e:
        logging.error(f"Error saving pipeline: {e}")
        exit()

    logging.info("Extended training script finished.")

2025-04-14 12:52:07,519 - INFO - Starting extended training script...
2025-04-14 12:52:07,525 - INFO - Loading data...
2025-04-14 12:52:07,625 - INFO - Data loaded successfully.
2025-04-14 12:52:07,626 - INFO - Preprocessing clinical notes...
2025-04-14 12:52:07,637 - INFO - Aggregated notes. Shape: (293, 2)
2025-04-14 12:52:07,637 - INFO - Preprocessing imaging data (metadata)...
2025-04-14 12:52:07,663 - INFO - Aggregated imaging features. Shape: (479, 5)
2025-04-14 12:52:07,663 - INFO - Joining datasets...
C:\Users\kedamm\AppData\Local\Temp\ipykernel_50280\19619904.py:121: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[co